# ResNet18 ex novo v6.1 - Waste Type Identification

Variante v6.1: parte dalla v6 (0.9701 BA) e aggiunge two-phase training + Focal Loss.

**Risultati v6 (0.9701 BA standard, 0.9739 BA TTA):**
- TTA 10-crop ha funzionato (+0.38% su standard), superando v3 con TTA
- Metal precision migliorata 0.81->0.90
- Problemi residui: Plastic recall 0.91, confusioni Plastic-Glass (10 campioni)

**Novita v6.1 rispetto a v6:**
- **Two-phase training (S:12)**: fase 1 (5 epoche) backbone congelato, allena solo classifier head su feature ImageNet stabili. Fase 2 sblocca backbone con LR differenziati.
- **Focal Loss (gamma=2.0, S:09)**: sostituisce CrossEntropyLoss. Up-pesa automaticamente gli esempi difficili (Plastic-Glass, Metal-Clothing) dove il modello e incerto.

Obiettivo: >= 0.975 BA standard, ridurre confusioni Plastic-Glass.

## 1. Mount Drive e clone repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import datetime
import subprocess
from getpass import getpass

REPO_OWNER = 'fabriziodrr'
REPO_NAME = 'ProjectML'
BASE_BRANCH = 'exp/resnet18-exnovo-v61'
RUN_ID = datetime.datetime.now().strftime('%Y%m%d-%H%M')
RUN_BRANCH = f'exp-resnet18-exnovo-v61-{RUN_ID}'
RUN_NAME = f'resnet18_exnovo_v61_{RUN_ID}'
REPO_PATH = f'/content/{REPO_NAME}'
PUBLIC_REPO_URL = f'https://github.com/{REPO_OWNER}/{REPO_NAME}.git'

def sanitize(text):
    if 'AUTH_REPO_URL' in globals():
        text = text.replace(AUTH_REPO_URL, 'https://***@github.com/' + REPO_OWNER + '/' + REPO_NAME + '.git')
    if 'GITHUB_TOKEN' in globals() and GITHUB_TOKEN:
        text = text.replace(GITHUB_TOKEN, '***')
    return text

def run(cmd, cwd=None, check=True):
    printable = sanitize(' '.join(cmd))
    print('$', printable)
    result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if result.stdout:
        print(sanitize(result.stdout))
    if result.stderr:
        print(sanitize(result.stderr))
    if check and result.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {result.returncode}: {printable}')
    return result

GITHUB_TOKEN = getpass('GitHub token personale con permessi repo: ')
AUTH_REPO_URL = f'https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git'

if not os.path.exists(REPO_PATH):
    run(['git', 'clone', '--branch', 'main', AUTH_REPO_URL, REPO_PATH])
else:
    print('Repo gia presente nel runtime.')

run(['git', 'remote', 'set-url', 'origin', AUTH_REPO_URL], cwd=REPO_PATH)
run(['git', 'pull', '--rebase', 'origin', 'main'], cwd=REPO_PATH)
run(['git', 'remote', 'set-url', 'origin', PUBLIC_REPO_URL], cwd=REPO_PATH)
branch_exists = subprocess.run(['git', 'rev-parse', '--verify', RUN_BRANCH], cwd=REPO_PATH, text=True, capture_output=True).returncode == 0
if branch_exists:
    run(['git', 'switch', RUN_BRANCH], cwd=REPO_PATH)
else:
    run(['git', 'switch', '-c', RUN_BRANCH], cwd=REPO_PATH)
os.chdir(REPO_PATH)
print('Working directory:', os.getcwd())
print('Run branch:', RUN_BRANCH)
print('Run name:', RUN_NAME)

## 2. Configurazione esperimento

In [ ]:
import json
import random
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedShuffleSplit
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import models, transforms
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

EXP_NAME = 'resnet18_exnovo_v61'
SEED = 42
NUM_CLASSES = 8
BATCH_SIZE = 32
NUM_WORKERS = 0
VAL_SIZE = 0.20
EPOCHS = 40
PATIENCE = 8
TWO_PHASE_FREEZE_EPOCHS = 5
FOCAL_GAMMA = 2.0
BACKBONE_LR = 1e-4
HEAD_LR = 5e-4
DROPOUT_P = 0.30
WEIGHT_DECAY = 2e-4
LABEL_SMOOTHING = 0.15
USE_AMP = True

DRIVE_ZIP_PATH = '/content/drive/MyDrive/waste_type_identification.zip'
DRIVE_DATASET_DIR = '/content/drive/MyDrive/waste_type_identification'
LOCAL_DATASET_DIR = '/content/waste_type_identification'

RESULTS_DIR = Path(REPO_PATH) / 'results' / RUN_NAME
DRIVE_CHECKPOINT_DIR = Path('/content/drive/MyDrive/ProjectML_checkpoints') / RUN_NAME

CLASS_NAMES = ['Battery', 'Clothing', 'Glass', 'Metal', 'Organic', 'Papery', 'Plastic', 'Undifferentiated']
LABEL_MAP = {
    'battery': 0,
    'clothes': 1, 'clothing': 1, 'shoes': 1,
    'brown': 2, 'green': 2, 'transparent': 2, 'glass': 2,
    'metal': 3,
    'organic': 4,
    'cardboard': 5, 'paper': 5, 'papery': 5,
    'plastic': 6,
    'undifferentiated': 7,
}
IMG_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp', '.gif', '.tiff', '.webp')

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

set_seed(SEED)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 3. Preparazione dataset

In [ ]:
local_dataset = Path(LOCAL_DATASET_DIR)
drive_dataset = Path(DRIVE_DATASET_DIR)
drive_zip = Path(DRIVE_ZIP_PATH)

if not local_dataset.exists():
    if drive_dataset.exists():
        print('Copio dataset gia estratto da Drive a disco locale Colab...')
        shutil.copytree(drive_dataset, local_dataset)
    elif drive_zip.exists():
        print('Estraggo zip dataset da Drive nella cartella locale attesa...')
        local_dataset.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(drive_zip, 'r') as zf:
            zf.extractall(local_dataset)
    else:
        raise FileNotFoundError(f'Non trovo ne {drive_dataset} ne {drive_zip}')

if not local_dataset.exists():
    candidates = [p for p in Path('/content').iterdir() if p.is_dir() and 'waste' in p.name.lower()]
    print('Candidate dataset folders:', candidates)
    raise FileNotFoundError(f'Dataset locale atteso non trovato: {local_dataset}')

print('Dataset locale:', local_dataset)
print('Prime cartelle:', sorted([p.name for p in local_dataset.iterdir() if p.is_dir()])[:30])

In [ ]:
samples = []
scanned_folders = set()
skipped_folders = set()
for dirpath, _, filenames in os.walk(local_dataset, followlinks=False):
    folder = Path(dirpath).name.lower()
    if not filenames:
        continue
    scanned_folders.add(folder)
    if folder not in LABEL_MAP:
        skipped_folders.add(folder)
        continue
    label = LABEL_MAP[folder]
    for fname in filenames:
        if fname.lower().endswith(IMG_EXTENSIONS):
            path = Path(dirpath) / fname
            samples.append({
                'path': str(path),
                'relative_path': str(path.relative_to(local_dataset)),
                'folder': folder,
                'label': label,
                'class_name': CLASS_NAMES[label],
            })

if skipped_folders:
    print(f'WARNING: {len(skipped_folders)} cartelle ignorate (non in LABEL_MAP): {sorted(skipped_folders)}')

df = pd.DataFrame(samples).sort_values('relative_path').reset_index(drop=True)
if df.empty:
    raise RuntimeError('Nessuna immagine trovata. Controlla LABEL_MAP e struttura dataset.')

print('Totale immagini:', len(df))
print(df['class_name'].value_counts().reindex(CLASS_NAMES, fill_value=0))

splitter = StratifiedShuffleSplit(n_splits=1, test_size=VAL_SIZE, random_state=SEED)
train_idx, val_idx = next(splitter.split(df['relative_path'], df['label']))
df['split'] = 'train'
df.loc[val_idx, 'split'] = 'val'

split_path = RESULTS_DIR / 'split.csv'
df[['relative_path', 'folder', 'label', 'class_name', 'split']].to_csv(split_path, index=False)
print('Split salvato in:', split_path)
print(pd.crosstab(df['class_name'], df['split']).reindex(CLASS_NAMES, fill_value=0))

## 4. Dataset, transform e DataLoader

In [ ]:
class WasteDataset(Dataset):
    def __init__(self, frame, transform=None):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        image = Image.open(row['path']).convert('RGB')
        if self.transform is not None:
            image = self.transform(image)
        return image, int(row['label'])

train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224, scale=(0.70, 1.0), ratio=(0.85, 1.18)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), shear=5),
    transforms.ColorJitter(brightness=0.30, contrast=0.25, saturation=0.20, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.10), ratio=(0.3, 3.3)),
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_df = df[df['split'] == 'train'].reset_index(drop=True)
val_df = df[df['split'] == 'val'].reset_index(drop=True)

train_labels = train_df['label'].to_numpy()
class_sample_counts = np.bincount(train_labels)
class_weights_sampler = 1.0 / class_sample_counts
sample_weights = class_weights_sampler[train_labels]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

train_loader = DataLoader(WasteDataset(train_df, train_transform), batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(WasteDataset(val_df, val_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

train_counts = train_df['label'].value_counts().reindex(range(NUM_CLASSES), fill_value=0).sort_index().to_numpy(dtype=np.float32)
class_weights = np.sqrt(train_counts.max() / np.maximum(train_counts, 1.0))
class_weights = class_weights / class_weights.mean()
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)

weights_df = pd.DataFrame({'label': range(NUM_CLASSES), 'class_name': CLASS_NAMES, 'train_count': train_counts.astype(int), 'loss_weight': class_weights})
weights_df.to_csv(RESULTS_DIR / 'class_weights.csv', index=False)
print(weights_df)
print('WeightedRandomSampler attivo: classi bilanciate nei batch')

## 5. Modello ResNet18 + Focal Loss

In [ ]:
class FocalLoss(nn.Module):
    """Focal Loss (S:09): down-weights easy examples, up-weights hard ones.
    FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)"""
    def __init__(self, weight=None, gamma=2.0, label_smoothing=0.0):
        super().__init__()
        self.weight = weight
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets, weight=self.weight,
                                 label_smoothing=self.label_smoothing, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)
num_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(p=DROPOUT_P),
    nn.Linear(num_features, NUM_CLASSES),
)
model = model.to(DEVICE)

criterion = FocalLoss(weight=class_weights_tensor, gamma=FOCAL_GAMMA, label_smoothing=LABEL_SMOOTHING)
backbone_params = [p for name, p in model.named_parameters() if not name.startswith('fc.')]
head_params = list(model.fc.parameters())
optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': BACKBONE_LR},
    {'params': head_params, 'lr': HEAD_LR},
], weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3, min_lr=1e-6)
scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP and DEVICE.type == 'cuda')

config = {
    'experiment': EXP_NAME,
    'run_id': RUN_ID,
    'run_name': RUN_NAME,
    'base_branch': BASE_BRANCH,
    'run_branch': RUN_BRANCH,
    'model': 'torchvision.models.resnet18 pretrained ImageNet',
    'strategy': 'two_phase_training_freeze_backbone_5epochs_then_full_finetuning_AdamW_discriminative_LR_FocalLoss_WeightedRandomSampler_LabelSmoothing_ReduceLROnPlateau_strongAug_TTA10crop',
    'seed': SEED,
    'val_size': VAL_SIZE,
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'patience': PATIENCE,
    'two_phase_freeze_epochs': TWO_PHASE_FREEZE_EPOCHS,
    'focal_gamma': FOCAL_GAMMA,
    'backbone_lr': BACKBONE_LR,
    'head_lr': HEAD_LR,
    'dropout_p': DROPOUT_P,
    'weight_decay': WEIGHT_DECAY,
    'label_smoothing': LABEL_SMOOTHING,
    'loss': 'FocalLoss(gamma=2.0) with sqrt inverse frequency class weights and label smoothing 0.15',
    'augmentation': 'Resize, aggressive RandomResizedCrop (0.70-1.0), horizontal flip, rotation 15deg, RandomAffine (translate+shear), stronger ColorJitter, RandomErasing p=0.25',
    'tta_augmentations': '10-crop: FiveCrop (tl, tr, bl, br, center) x horizontal flip',
    'class_balancing': 'WeightedRandomSampler with inverse frequency weights',
    'regularization': 'Dropout(0.30) + L2(2e-4) + LabelSmoothing(0.15) + TwoPhaseTraining',
    'course_arguments': 'transfer learning (S:12), two-phase/freeze training (S:12), discriminative LRs (S:12), WeightedRandomSampler (S:05), label smoothing (S:09), focal loss (S:09), ReduceLROnPlateau (S:09), early stopping (S:09), L2/weight_decay (S:09), dropout (S:13), data augmentation (S:05), TTA 10-crop (S:05), balanced accuracy (S:05), StratifiedShuffleSplit (L:01)',
    'dataset_drive_zip': DRIVE_ZIP_PATH,
    'dataset_drive_dir': DRIVE_DATASET_DIR,
    'results_dir': str(RESULTS_DIR),
    'checkpoint_drive_dir': str(DRIVE_CHECKPOINT_DIR),
}
with open(RESULTS_DIR / 'config.json', 'w') as f:
    json.dump(config, f, indent=2)
print(json.dumps(config, indent=2))

## 6. Two-phase training (freeze backbone -> full finetuning) + ReduceLROnPlateau

In [ ]:
def freeze_backbone(model, freeze):
    """Freeze/unfreeze all backbone parameters. Head (fc) always trainable."""
    for name, param in model.named_parameters():
        if not name.startswith('fc.'):
            param.requires_grad = not freeze

def run_epoch(model, loader, train):
    model.train(train)
    total_loss = 0.0
    all_preds = []
    all_targets = []

    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for images, targets in loader:
            images = images.to(DEVICE, non_blocking=True)
            targets = targets.to(DEVICE, non_blocking=True)

            if train:
                optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP and DEVICE.type == 'cuda'):
                logits = model(images)
                loss = criterion(logits, targets)

            if train:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

            total_loss += loss.item() * images.size(0)
            all_preds.append(logits.argmax(dim=1).detach().cpu())
            all_targets.append(targets.detach().cpu())

    y_pred = torch.cat(all_preds).numpy()
    y_true = torch.cat(all_targets).numpy()
    loss = total_loss / len(loader.dataset)
    acc = float((y_pred == y_true).mean())
    bal_acc = float(balanced_accuracy_score(y_true, y_pred))
    return loss, acc, bal_acc, y_true, y_pred

def evaluate_with_tta(model, frame):
    """Evaluate with 10-crop TTA: FiveCrop (5 crops) x HorizontalFlip (2 flips) = 10 views."""
    model.eval()
    y_true, y_pred = [], []
    total = len(frame)

    crop_224 = transforms.FiveCrop(224)
    normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    to_tensor = transforms.ToTensor()

    for idx in range(total):
        row = frame.iloc[idx]
        image = Image.open(row["path"]).convert("RGB")

        probs_list = []
        image_256 = transforms.Resize(256)(image)
        crops = crop_224(image_256)
        for crop in crops:
            crop_tensor = normalize(to_tensor(crop))
            crop_tensor = crop_tensor.unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                probs_list.append(F.softmax(model(crop_tensor), dim=1))

            crop_flipped = transforms.functional.hflip(crop)
            crop_f_tensor = normalize(to_tensor(crop_flipped))
            crop_f_tensor = crop_f_tensor.unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                probs_list.append(F.softmax(model(crop_f_tensor), dim=1))

        avg_probs = torch.stack(probs_list).mean(dim=0)
        pred = avg_probs.argmax(dim=1).item()
        y_true.append(int(row["label"]))
        y_pred.append(pred)
        if (idx + 1) % 500 == 0:
            print(f"  TTA progress: {idx+1}/{total}")
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    acc = float((y_pred == y_true).mean())
    bal_acc = float(balanced_accuracy_score(y_true, y_pred))
    return acc, bal_acc, y_true, y_pred

best_bal_acc = -1.0
best_epoch = -1
epochs_without_improvement = 0
history = []
best_checkpoint_path = DRIVE_CHECKPOINT_DIR / 'resnet18_exnovo_v61_best.pth'

# Phase 1: freeze backbone, train only head
freeze_backbone(model, freeze=True)
print(f'Phase 1: backbone frozen for {TWO_PHASE_FREEZE_EPOCHS} epochs (training head only)')
for epoch in range(1, TWO_PHASE_FREEZE_EPOCHS + 1):
    train_loss, train_acc, train_bal_acc, _, _ = run_epoch(model, train_loader, train=True)
    val_loss, val_acc, val_bal_acc, y_true, y_pred = run_epoch(model, val_loader, train=False)

    row = {
        'epoch': epoch, 'phase': 'freeze',
        'backbone_lr': 0.0, 'head_lr': optimizer.param_groups[1]['lr'],
        'train_loss': train_loss, 'train_acc': train_acc, 'train_bal_acc': train_bal_acc,
        'val_loss': val_loss, 'val_acc': val_acc, 'val_bal_acc': val_bal_acc,
    }
    history.append(row)
    print(f'Epoch {epoch:02d} [freeze]: lr=[{row["backbone_lr"]:.2e},{row["head_lr"]:.2e}] train_loss={train_loss:.4f} train_ba={train_bal_acc:.4f} val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_ba={val_bal_acc:.4f}')

    if val_bal_acc > best_bal_acc:
        best_bal_acc = val_bal_acc
        best_epoch = epoch
        torch.save({
            'model_state_dict': model.state_dict(),
            'class_names': CLASS_NAMES,
            'config': config,
            'best_epoch': best_epoch,
            'best_val_bal_acc': best_bal_acc,
        }, best_checkpoint_path)
        print('  Saved best checkpoint to Drive:', best_checkpoint_path)

    scheduler.step(val_bal_acc)

# Phase 2: unfreeze backbone, full finetuning with ReduceLROnPlateau
freeze_backbone(model, freeze=False)
epochs_without_improvement = 0
print(f'\nPhase 2: backbone UNFROZEN, full finetuning from epoch {TWO_PHASE_FREEZE_EPOCHS + 1}')
for epoch in range(TWO_PHASE_FREEZE_EPOCHS + 1, EPOCHS + 1):
    train_loss, train_acc, train_bal_acc, _, _ = run_epoch(model, train_loader, train=True)
    val_loss, val_acc, val_bal_acc, y_true, y_pred = run_epoch(model, val_loader, train=False)

    row = {
        'epoch': epoch, 'phase': 'finetune',
        'backbone_lr': optimizer.param_groups[0]['lr'],
        'head_lr': optimizer.param_groups[1]['lr'],
        'train_loss': train_loss, 'train_acc': train_acc, 'train_bal_acc': train_bal_acc,
        'val_loss': val_loss, 'val_acc': val_acc, 'val_bal_acc': val_bal_acc,
    }
    history.append(row)

    print(f'Epoch {epoch:02d} [finetune]: lr=[{row["backbone_lr"]:.2e},{row["head_lr"]:.2e}] train_loss={train_loss:.4f} train_ba={train_bal_acc:.4f} val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_ba={val_bal_acc:.4f}')

    if val_bal_acc > best_bal_acc:
        best_bal_acc = val_bal_acc
        best_epoch = epoch
        epochs_without_improvement = 0
        torch.save({
            'model_state_dict': model.state_dict(),
            'class_names': CLASS_NAMES,
            'config': config,
            'best_epoch': best_epoch,
            'best_val_bal_acc': best_bal_acc,
        }, best_checkpoint_path)
        print('  Saved best checkpoint to Drive:', best_checkpoint_path)
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print('Early stopping: balanced accuracy non migliora da', PATIENCE, 'epoche.')
            break

    scheduler.step(val_bal_acc)

pd.DataFrame(history).to_csv(RESULTS_DIR / 'history.csv', index=False)

summary = {'run_name': RUN_NAME, 'run_branch': RUN_BRANCH, 'results_dir': str(RESULTS_DIR), 'best_epoch': best_epoch, 'best_val_bal_acc': best_bal_acc, 'checkpoint_path_drive': str(best_checkpoint_path)}
with open(RESULTS_DIR / 'summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('Summary:', summary)

## 7. Metriche finali e grafici (standard + TTA 10-crop)

In [ ]:
checkpoint = torch.load(best_checkpoint_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

val_loss, val_acc, val_bal_acc, y_true, y_pred = run_epoch(model, val_loader, train=False)
report = classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0)
cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))

with open(RESULTS_DIR / 'classification_report.txt', 'w') as f:
    f.write(report)
pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(RESULTS_DIR / 'confusion_matrix.csv')

print(report)
print('Final val accuracy:', val_acc)
print('Final val balanced accuracy:', val_bal_acc)

print('\nRunning TTA 10-crop evaluation on validation set...')
tta_acc, tta_bal_acc, y_true_tta, y_pred_tta = evaluate_with_tta(model, val_df)
tta_report = classification_report(y_true_tta, y_pred_tta, target_names=CLASS_NAMES, zero_division=0)
tta_cm = confusion_matrix(y_true_tta, y_pred_tta, labels=list(range(NUM_CLASSES)))

print('\n--- TTA 10-crop Results ---')
print(tta_report)
print('TTA accuracy:', tta_acc)
print('TTA balanced accuracy:', tta_bal_acc)

with open(RESULTS_DIR / 'classification_report_tta.txt', 'w') as f:
    f.write(tta_report)
pd.DataFrame(tta_cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(RESULTS_DIR / 'confusion_matrix_tta.csv')

with open(RESULTS_DIR / 'summary.json', 'r') as f:
    summary_data = json.load(f)
summary_data['tta_val_acc'] = tta_acc
summary_data['tta_val_bal_acc'] = tta_bal_acc
with open(RESULTS_DIR / 'summary.json', 'w') as f:
    json.dump(summary_data, f, indent=2)


history_df = pd.read_csv(RESULTS_DIR / 'history.csv')
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history_df['epoch'], history_df['train_loss'], label='train')
plt.plot(history_df['epoch'], history_df['val_loss'], label='val')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('ResNet18 v6.1 loss')
plt.grid(alpha=0.3)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_df['epoch'], history_df['train_bal_acc'], label='train BA')
plt.plot(history_df['epoch'], history_df['val_bal_acc'], label='val BA')
plt.xlabel('Epoch')
plt.ylabel('Balanced accuracy')
plt.title('ResNet18 v6.1 balanced accuracy')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'training_curves.png', dpi=160)
plt.show()

plt.figure(figsize=(8, 7))
plt.imshow(cm, interpolation='nearest', cmap='Blues')
plt.title('ResNet18 v6.1 confusion matrix')
plt.colorbar()
ticks = np.arange(NUM_CLASSES)
plt.xticks(ticks, CLASS_NAMES, rotation=45, ha='right')
plt.yticks(ticks, CLASS_NAMES)
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        plt.text(j, i, cm[i, j], ha='center', va='center', color='white' if cm[i, j] > cm.max() / 2 else 'black')
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'confusion_matrix.png', dpi=160)
plt.show()

cm_norm = cm.astype('float64') / cm.sum(axis=1, keepdims=True)
cm_norm = np.nan_to_num(cm_norm)
plt.figure(figsize=(8, 7))
plt.imshow(cm_norm, interpolation='nearest', cmap='Greens', vmin=0, vmax=1)
plt.title('ResNet18 v6.1 confusion matrix (normalized by row)')
plt.colorbar()
plt.xticks(ticks, CLASS_NAMES, rotation=45, ha='right')
plt.yticks(ticks, CLASS_NAMES)
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        plt.text(j, i, f'{cm_norm[i, j]:.2f}', ha='center', va='center', fontsize=8,
                 color='white' if cm_norm[i, j] > 0.5 else 'black')
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'confusion_matrix_norm.png', dpi=160)
plt.show()

plt.figure(figsize=(8, 7))
plt.imshow(tta_cm, interpolation="nearest", cmap="Oranges")
plt.title('ResNet18 v6.1 TTA 10-crop confusion matrix')
plt.colorbar()
plt.xticks(ticks, CLASS_NAMES, rotation=45, ha="right")
plt.yticks(ticks, CLASS_NAMES)
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        plt.text(j, i, tta_cm[i, j], ha="center", va="center", color="white" if tta_cm[i, j] > tta_cm.max() / 2 else "black")
plt.ylabel("True label")
plt.xlabel("Predicted label")
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'confusion_matrix_tta.png', dpi=160)
plt.show()

tta_cm_norm = tta_cm.astype("float64") / tta_cm.sum(axis=1, keepdims=True)
tta_cm_norm = np.nan_to_num(tta_cm_norm)
plt.figure(figsize=(8, 7))
plt.imshow(tta_cm_norm, interpolation='nearest', cmap='Reds', vmin=0, vmax=1)
plt.title('ResNet18 v6.1 TTA 10-crop confusion matrix (normalized by row)')
plt.colorbar()
plt.xticks(ticks, CLASS_NAMES, rotation=45, ha='right')
plt.yticks(ticks, CLASS_NAMES)
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        plt.text(j, i, f'{tta_cm_norm[i, j]:.2f}', ha='center', va='center', fontsize=8,
                 color='white' if tta_cm_norm[i, j] > 0.5 else 'black')
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'confusion_matrix_tta_norm.png', dpi=160)
plt.show()


## 8. Note esperimento

In [ ]:
notes = f'''# {RUN_NAME}

## Ipotesi
La v6.1 parte dalla v6 e aggiunge two-phase training + Focal Loss per attaccare i punti deboli residui: overfitting e confusioni Plastic-Glass / Metal-Clothing.

## Novita v6.1 rispetto a v6
1. **Two-phase training (S:12)**:
   - Fase 1 (5 epoche): backbone congelato, allena solo il classifier head su feature ImageNet stabili.
   - Fase 2: sblocca backbone, full finetuning con LR differenziati + ReduceLROnPlateau.
   - Vantaggio: la head impara a discriminare senza che il backbone si adatti precocemente, riducendo overfitting.
2. **Focal Loss (gamma=2.0, S:09)**:
   - Sostituisce CrossEntropyLoss. FL(p_t) = -(1-p_t)^gamma * log(p_t)
   - Down-pesa esempi facili, up-pesa automaticamente quelli incerti (Plastic-Glass, Metal-Clothing).

## Ereditato da v6
- Rimosso warmup LR, patience 8, RLRP patience 3
- Dropout 0.30, label smoothing 0.15, weight decay 2e-4
- Augmentation forte: crop 0.70-1.0, rotation 15deg, RandomAffine, ColorJitter 0.30/0.25/0.20, RandomErasing p=0.25
- TTA 10-crop (FiveCrop x HorizontalFlip)

## Cosa ha funzionato (da analisi v2, v3, v3.2, v4, v5, v6)
- **WeightedRandomSampler**: +1.34% BA rispetto a v2
- **Label smoothing 0.1**: calibrazione migliorata
- **ReduceLROnPlateau**: decadimento adattivo superiore a CosineAnnealingLR
- **AdamW con LR differenziati**: convergenza rapida e stabile
- **ResNet18**: bilanciamento ottimale capacita/overfitting

## Tecniche del corso usate
- Transfer learning (S:12): ResNet18 pretrained ImageNet
- Two-phase training / freeze (S:12): backbone congelato poi sbloccato
- Discriminative LRs (S:12): backbone piu conservativo
- WeightedRandomSampler (S:05): class balancing
- Label smoothing (S:09): calibrazione
- Focal Loss (S:09): loss adattiva per esempi difficili
- ReduceLROnPlateau (S:09): adaptive LR decay
- Dropout 0.30 (S:13, L:07A): regolarizzazione
- L2/weight_decay 2e-4 (S:09): regolarizzazione
- Data augmentation (S:05, L:04): flip, rotazione, Affine, ColorJitter, RandomErasing
- TTA 10-crop (S:05): 10 viste in inference
- Early stopping (S:09, L:04): patience=8
- Balanced accuracy (S:05): metrica primaria
- StratifiedShuffleSplit (L:01): split 80/20

## Target
- Standard BA >= 0.975 (superare v3 0.9725 e v6 0.9701)
- TTA 10-crop: ulteriore +0.2-0.5% atteso
- Ridurre confusioni Plastic-Glass e Metal-Clothing

## Risultato
Best epoch: {best_epoch}
Best validation balanced accuracy: {best_bal_acc:.4f}
TTA validation accuracy: {tta_acc:.4f}
TTA validation balanced accuracy: {tta_bal_acc:.4f}
Branch run: {RUN_BRANCH}
Cartella risultati: {RESULTS_DIR}
Checkpoint Drive: {best_checkpoint_path}

## File prodotti in repo
- `split.csv`
- `config.json`
- `class_weights.csv`
- `history.csv`
- `summary.json`
- `classification_report.txt`
- `classification_report_tta.txt`
- `confusion_matrix.csv`
- `confusion_matrix.png`
- `confusion_matrix_norm.png`
- `confusion_matrix_tta.csv`
- `confusion_matrix_tta.png`
- `confusion_matrix_tta_norm.png`
- `training_curves.png`
'''
with open(RESULTS_DIR / 'notes.md', 'w') as f:
    f.write(notes)
print(notes)


## 9. Commit e push risultati su GitHub

In [ ]:
os.chdir(REPO_PATH)
run(['git', 'config', 'user.name', 'fabriziodrr'], cwd=REPO_PATH)
run(['git', 'config', 'user.email', 'fabriziodrr@users.noreply.github.com'], cwd=REPO_PATH)

run(['git', 'add', str(RESULTS_DIR.relative_to(REPO_PATH))], cwd=REPO_PATH)
status = subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO_PATH).decode().strip()

if not status:
    print('Nessuna modifica da committare.')
else:
    message = f'Add ResNet18 exnovo v6.1 results {RUN_ID}'
    run(['git', 'commit', '-m', message], cwd=REPO_PATH)
    auth_url = f'https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git'
    try:
        run(['git', 'remote', 'set-url', 'origin', auth_url], cwd=REPO_PATH)
        run(['git', 'push', '-u', 'origin', RUN_BRANCH], cwd=REPO_PATH)
    finally:
        run(['git', 'remote', 'set-url', 'origin', PUBLIC_REPO_URL], cwd=REPO_PATH, check=False)

print('Risultati sincronizzati su GitHub.')